# Metro platform crowd density — training on Colab

Fine-tunes YOLO on CrowdHuman for dense person detection, and compares it
against the untouched pretrained baseline.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **GPU**.

Run the cells in order. Section 8 saves everything you need back to Drive,
including the results table you commit to your repository.


## 1. Check the GPU

If this says `no GPU`, stop and change the runtime type first — training on CPU will not finish.


### Step 0 — Check the hardware

**What this does:** confirms Colab gave us a GPU. Training a vision model on a CPU would take days, so this is a stop-and-check before anything else.


In [ ]:
import torch, subprocess
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(subprocess.run(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip())
else:
    print('no GPU — go to Runtime > Change runtime type > GPU, then restart')


## 2. Install



### Step 1 — Install the tools

**What this does:** installs Ultralytics (the YOLO library) and the Hugging Face downloader. Nothing clever here — just getting the software we need.


In [ ]:
!pip install -q ultralytics huggingface_hub
import ultralytics; ultralytics.checks()


## 3. Mount Drive

Used to save your trained model and results so they survive the session ending.
Colab wipes everything else when it disconnects.


### Step 2 — Connect Google Drive

**What this does:** links Drive so the trained model and results can be saved somewhere permanent. Colab erases everything when it disconnects; Drive keeps our work.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
SAVE_DIR = Path('/content/drive/MyDrive/metro-crowd-detection')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('saving to', SAVE_DIR)


## 4. Get CrowdHuman

Colab's connection is fast, so this downloads in minutes rather than hours.

**Start with `VAL_ONLY = True`.** That pulls the 4,370-image validation set only,
so you can check the whole pipeline works before committing to the full download.
Set it to `False` once you have confirmed everything runs.


### Step 3 — Data collection

**What this does:** downloads CrowdHuman, the dataset of crowd photos with every person already boxed by hand. This is our raw material. We start with the validation part only, to check everything works before pulling the full set.


In [ ]:
VAL_ONLY = True   # set False for the full training set

from huggingface_hub import snapshot_download

patterns = ['*.odgt', 'CrowdHuman_val*'] if VAL_ONLY else None

path = snapshot_download(
    repo_id='sshao0516/CrowdHuman',
    repo_type='dataset',
    local_dir='/content/chraw',
    allow_patterns=patterns,
)
print('downloaded to', path)
!ls -la /content/chraw | head -20


### Unpack



### Step 4 — Unpack the data

**What this does:** the download arrives as zip files. This unzips the images and puts the annotation files where the next step expects them.


In [ ]:
import zipfile, glob, os, shutil
from pathlib import Path

RAW = Path('/content/chraw')
SRC = Path('/content/crowdhuman')
SRC.mkdir(exist_ok=True)

for odgt in RAW.rglob('*.odgt'):
    shutil.copy2(odgt, SRC / odgt.name)

def unzip_into(pattern, target):
    target.mkdir(parents=True, exist_ok=True)
    for z in sorted(RAW.rglob(pattern)):
        print('unzipping', z.name)
        with zipfile.ZipFile(z) as zf:
            zf.extractall('/content/_tmp_unzip')
    for p in Path('/content/_tmp_unzip').rglob('*'):
        if p.suffix.lower() in {'.jpg','.jpeg','.png'}:
            dest = target / p.name
            if not dest.exists():
                shutil.move(str(p), dest)
    shutil.rmtree('/content/_tmp_unzip', ignore_errors=True)

unzip_into('CrowdHuman_val*.zip', SRC / 'Images_val')
if not VAL_ONLY:
    unzip_into('CrowdHuman_train*.zip', SRC / 'Images_train')

for name in ['Images_train','Images_val']:
    d = SRC / name
    n = len(list(d.glob('*.jpg'))) if d.exists() else 0
    print(f'{name}: {n} images')
print([p.name for p in SRC.glob('*.odgt')])


## 5. Convert to YOLO format

CrowdHuman gives every person three boxes. We use **`hbox`** (the head), because
station cameras are mounted high and look down — heads stay visible in a crowd
when bodies are hidden behind other bodies.

This is the same converter as in your repository, inlined so the notebook runs standalone.


### Step 5 — Annotation conversion, normalization and cleaning

This is the heart of the data preparation. Three things happen here.

**Annotation conversion:** CrowdHuman's labels come in its own JSON format. YOLO needs a different format — one text file per image. We translate between them.

**Coordinate normalization (scaling):** box positions are given in pixels, like x = 350. We divide by the image size to turn them into fractions between 0 and 1, like 0.44. This is the scaling step — it lets the model work no matter the image size.

**Data cleaning:** boxes marked 'ignore' are removed, boxes hanging off the edge of the image are trimmed to fit, and any box that ends up with zero size is dropped.

**Label choice:** we keep the **head** box for each person, not the full body, because metro cameras look down from above and heads are what stays visible in a crowd.


In [ ]:
import json
from pathlib import Path
from PIL import Image

def clamp(v, lo=0.0, hi=1.0):
    return max(lo, min(hi, v))

def to_yolo_line(box, iw, ih):
    x, y, w, h = box
    if w <= 0 or h <= 0:
        return None
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(iw, x + w), min(ih, y + h)
    if x2 <= x1 or y2 <= y1:
        return None
    xc = clamp(((x1 + x2) / 2) / iw); yc = clamp(((y1 + y2) / 2) / ih)
    bw = clamp((x2 - x1) / iw);       bh = clamp((y2 - y1) / ih)
    if bw <= 0 or bh <= 0:
        return None
    return f'0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}'

def convert(odgt, images_dir, out_dir, box_key='hbox', limit=None):
    out_img = Path(out_dir) / 'images'; out_lab = Path(out_dir) / 'labels'
    out_img.mkdir(parents=True, exist_ok=True); out_lab.mkdir(parents=True, exist_ok=True)
    written = kept = dropped = missing = 0
    with open(odgt, encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line or (limit and written >= limit):
                continue
            rec = json.loads(line)
            src = None
            for ext in ('.jpg','.jpeg','.png'):
                cand = Path(images_dir) / f"{rec['ID']}{ext}"
                if cand.exists():
                    src = cand; break
            if src is None:
                missing += 1; continue
            try:
                with Image.open(src) as im:
                    iw, ih = im.size
            except Exception:
                missing += 1; continue
            lines = []
            for gt in rec.get('gtboxes', []):
                if gt.get('tag') != 'person':
                    continue
                if gt.get('extra', {}).get('ignore', 0) or gt.get('head_attr', {}).get('ignore', 0):
                    continue
                box = gt.get(box_key)
                if not box or len(box) != 4:
                    continue
                ln = to_yolo_line(box, iw, ih)
                if ln is None:
                    dropped += 1
                else:
                    lines.append(ln); kept += 1
            (out_lab / f"{rec['ID']}.txt").write_text('\n'.join(lines))
            dst = out_img / src.name
            if not dst.exists():
                try:    dst.symlink_to(src.resolve())
                except Exception: shutil.copy2(src, dst)
            written += 1
    print(f'  images {written} | boxes {kept} | dropped {dropped} | missing {missing}')
    if written: print(f'  mean people per image: {kept/written:.1f}')
    return written

DATA = Path('/content/data')
BOX = 'hbox'

print('val:')
n_val = convert(SRC/'annotation_val.odgt', SRC/'Images_val', DATA/'val', BOX)

if not VAL_ONLY and (SRC/'Images_train').exists():
    print('train:')
    n_train = convert(SRC/'annotation_train.odgt', SRC/'Images_train', DATA/'train', BOX)
else:
    # no train images yet: split val so the pipeline still runs end to end
    print('no training images — splitting val into train/val for a pipeline test')
    import random
    random.seed(0)
    imgs = sorted((DATA/'val'/'images').glob('*'))
    random.shuffle(imgs)
    cut = int(len(imgs) * 0.8)
    (DATA/'train'/'images').mkdir(parents=True, exist_ok=True)
    (DATA/'train'/'labels').mkdir(parents=True, exist_ok=True)
    for p in imgs[:cut]:
        shutil.move(str(p), DATA/'train'/'images'/p.name)
        lab = DATA/'val'/'labels'/(p.stem + '.txt')
        if lab.exists(): shutil.move(str(lab), DATA/'train'/'labels'/lab.name)
    print('train', len(list((DATA/'train'/'images').glob('*'))),
          '| val', len(list((DATA/'val'/'images').glob('*'))))


### Look at the boxes before training

Numbers can look right while boxes sit in the wrong place. Always check visually.
Boxes should be on **heads**.


### Step 6 — Look at the data (validation)

**What this does:** draws the boxes back onto a few images so we can see them with our own eyes. Numbers can look fine while boxes sit in the wrong place — so we check that they actually land on heads before wasting hours on training.


In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image

imgs = sorted((DATA/'train'/'images').glob('*'))[:2]
for ip in imgs:
    lp = DATA/'train'/'labels'/(ip.stem + '.txt')
    if not lp.exists(): continue
    im = Image.open(ip); W, H = im.size
    fig, ax = plt.subplots(figsize=(10,7)); ax.imshow(im)
    n = 0
    for ln in lp.read_text().splitlines():
        p = ln.split()
        if len(p) != 5: continue
        _, xc, yc, w, h = map(float, p)
        ax.add_patch(patches.Rectangle(((xc-w/2)*W, (yc-h/2)*H), w*W, h*H,
                                       fill=False, edgecolor='lime', linewidth=1.4))
        n += 1
    ax.set_title(f'{ip.name} — {n} people'); ax.axis('off'); plt.show()


### Dataset config



### Step 7 — Point YOLO at the data (the train/validation split)

**What this does:** writes a small config file telling YOLO where the training images are and where the separate validation images are. Keeping these two sets apart is what makes the test honest — the model is never checked on images it learned from.


In [ ]:
yaml_text = f'''path: {DATA}
train: train/images
val: val/images

names:
  0: person
'''
Path('/content/crowdhuman.yaml').write_text(yaml_text)
print(yaml_text)


## 6. Measure the baseline FIRST

This is your **before** number. Run it before training anything — it is much
harder to reconstruct later, and the whole project rests on this comparison.

The baseline is untouched pretrained YOLO. It is never modified.


### Step 8 — Measure the baseline first

**What this does:** runs the plain, untrained YOLO on our test images and records its scores. This is the 'before' picture. We do it *before* training, because the whole project is about showing the 'after' beats the 'before'.

Note: image preprocessing (resizing every image to 960 pixels and scaling the pixel values) happens automatically inside YOLO here — we don't write it.


In [ ]:
import math, json
from datetime import datetime
from ultralytics import YOLO

RESULTS = Path('/content/results'); RESULTS.mkdir(exist_ok=True)
CONF = 0.35
BASELINE = 'yolov8n.pt'

def counting_error(model, images_dir, labels_dir, conf=CONF, limit=300):
    paths = sorted(p for p in Path(images_dir).iterdir()
                   if p.suffix.lower() in {'.jpg','.jpeg','.png'})[:limit]
    abs_e, sq_e, n = [], [], 0
    for p in paths:
        lab = Path(labels_dir) / (p.stem + '.txt')
        if not lab.exists(): continue
        true_n = sum(1 for l in lab.read_text().splitlines() if l.strip())
        pred_n = len(model.predict(source=str(p), conf=conf, classes=[0], verbose=False)[0].boxes)
        d = pred_n - true_n
        abs_e.append(abs(d)); sq_e.append(d*d); n += 1
    if not n: return {}
    return {'images_compared': n,
            'count_mae': round(sum(abs_e)/n, 3),
            'count_rmse': round(math.sqrt(sum(sq_e)/n), 3)}

def record(name, weights, conf=CONF):
    m = YOLO(weights)
    mt = m.val(data='/content/crowdhuman.yaml', conf=conf, verbose=False)
    e = {'name': name, 'weights': str(weights), 'confidence': conf,
         'map50': round(float(mt.box.map50), 4),
         'map50_95': round(float(mt.box.map), 4),
         'precision': round(float(mt.box.mp), 4),
         'recall': round(float(mt.box.mr), 4),
         'evaluated_at': datetime.now().isoformat(timespec='seconds')}
    e.update(counting_error(m, DATA/'val'/'images', DATA/'val'/'labels', conf))
    f = RESULTS/'results.json'
    entries = json.loads(f.read_text()) if f.exists() else []
    entries = [x for x in entries if x['name'] != name] + [e]
    entries.sort(key=lambda x: x['name'])
    f.write_text(json.dumps(entries, indent=2))
    write_table(entries)
    print(json.dumps(e, indent=2))
    return e

def write_table(entries):
    head = ('| Model | mAP@0.5 | Precision | Recall | Count MAE | Count RMSE | Images |\n'
            '|---|---|---|---|---|---|---|\n')
    rows = ''
    for e in entries:
        rows += (f"| {e['name']} | {e.get('map50','-')} | {e.get('precision','-')} | "
                 f"{e.get('recall','-')} | {e.get('count_mae','-')} | "
                 f"{e.get('count_rmse','-')} | {e.get('images_compared','-')} |\n")
    (RESULTS/'results.md').write_text(
        '# Model comparison\n\nLower counting error is better. '
        'Higher mAP, precision and recall are better.\n\n' + head + rows +
        '\nAll models evaluated on the same held-out data at the same confidence threshold.\n')

baseline_scores = record('baseline', BASELINE)


## 7. Train

Settings chosen for this problem:

- `imgsz=960` — distant heads survive. At the default 640 they shrink to a few pixels and vanish.
- `hbox` labels — already applied during conversion.
- `epochs=30` — reduce to 10 for a quick check first.

The trained model is saved as a **new file**. The baseline is never overwritten,
so the comparison stays valid.


### Step 9 — Training (transfer learning)

**What this does:** this is the actual learning. We don't start from nothing — we take a model already trained on everyday photos and continue training it on our crowd heads. That's called transfer learning, and it's why an hour is enough instead of weeks.

During training YOLO also augments the images — random flips, zooms, and stitching four photos into one — so the model sees more variety and learns better. The trained model is saved as a new file; the baseline is never touched.


In [ ]:
RUN_NAME = 'run1'
EPOCHS   = 30
IMGSZ    = 960
BATCH    = 8      # lower this if you hit an out-of-memory error
BASE     = 'yolov8n.pt'   # try yolov8s.pt if GPU memory allows

assert RUN_NAME not in {'yolov8n','yolov8s','yolov8m'}, 'do not use a baseline name'

model = YOLO(BASE)
model.train(data='/content/crowdhuman.yaml', epochs=EPOCHS, imgsz=IMGSZ,
            batch=BATCH, name=RUN_NAME, project='/content/runs', exist_ok=False)

best = Path(f'/content/runs/{RUN_NAME}/weights/best.pt')
print('trained weights:', best, best.exists())


## 8. Measure the trained model

Same data, same confidence threshold, same metrics. Only the model changed.


### Step 10 — Measure the trained model (evaluation)

**What this does:** runs the newly trained model on the same test images, the same way, and records its scores next to the baseline. Same data, same settings — only the model changed. This is the comparison the whole project rests on.


In [ ]:
trained_scores = record(RUN_NAME, str(best))

print()
print((RESULTS/'results.md').read_text())


### Before and after



### Step 11 — Before vs after

**What this does:** prints the two sets of scores side by side so the improvement is easy to read — did recall go up, did the counting error go down.


In [ ]:
b, t = baseline_scores, trained_scores
def delta(k, higher_better=True):
    if k not in b or k not in t: return ''
    d = t[k] - b[k]
    good = (d > 0) if higher_better else (d < 0)
    return f"{b[k]:.3f} -> {t[k]:.3f}  ({d:+.3f}) {'better' if good else 'worse'}"

print('mAP@0.5   ', delta('map50'))
print('recall    ', delta('recall'))
print('precision ', delta('precision'))
print('count MAE ', delta('count_mae', higher_better=False))


## 9. Visual comparison on your own photos

Upload a Tashkent metro photo. Same image, both models, side by side.
This is the single most persuasive thing in your defence — save the output.


### Step 12 — See it on real photos

**What this does:** you upload a real platform photo and it shows the baseline and your model side by side, with both counts. This is the picture for your report — the visible proof that training worked.


In [ ]:
from google.colab import files
import matplotlib.pyplot as plt

uploaded = files.upload()

base_m = YOLO(BASELINE)
fine_m = YOLO(str(best))

for fname in uploaded:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    for ax, (label, m) in zip(axes, [('baseline', base_m), ('fine-tuned', fine_m)]):
        r = m.predict(source=fname, conf=CONF, classes=[0], verbose=False)[0]
        ax.imshow(r.plot()[:, :, ::-1])
        ax.set_title(f'{label} — {len(r.boxes)} people', fontsize=15)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'/content/results/compare_{Path(fname).stem}.png', dpi=110, bbox_inches='tight')
    plt.show()


## 10. Save everything to Drive

Colab deletes everything when the session ends. This copies out what you need:

- `finetuned.pt` — put this in your repo at `backend/weights/finetuned.pt` so the demo uses it
- `results.md` and `results.json` — **commit these to git**, they are your evidence
- the comparison images — for your README and slides


### Step 13 — Save everything

**What this does:** copies the trained model, the results table and the comparison images to Google Drive, so nothing is lost when Colab closes.


In [ ]:
import shutil

(SAVE_DIR/'weights').mkdir(parents=True, exist_ok=True)
(SAVE_DIR/'results').mkdir(parents=True, exist_ok=True)

shutil.copy2(best, SAVE_DIR/'weights'/'finetuned.pt')
shutil.copy2(best, SAVE_DIR/'weights'/f'{RUN_NAME}.pt')

for p in RESULTS.iterdir():
    shutil.copy2(p, SAVE_DIR/'results'/p.name)

train_dir = Path(f'/content/runs/{RUN_NAME}')
for name in ['results.png','confusion_matrix.png','results.csv','args.yaml']:
    src = train_dir / name
    if src.exists():
        shutil.copy2(src, SAVE_DIR/'results'/f'train_{name}')

print('saved to', SAVE_DIR)
for p in sorted(SAVE_DIR.rglob('*')):
    if p.is_file(): print(' ', p.relative_to(SAVE_DIR))


## What to do next

1. Download `finetuned.pt` from Drive into your repo at `backend/weights/finetuned.pt`.
   The demo dropdown then compares baseline against your model for real.
2. Commit `results/results.md` and `results.json` to git. Do **not** commit the `.pt` files — too large.
3. Commit this notebook to `notebooks/` so the training is reproducible, as your course requires.
4. Write the numbers into your report, plus an honest note on where the model fails
   (very dense far-field crowds are undercounted).

**Do not** rerun section 6 after training — it would overwrite your baseline row
with an identical value and waste GPU time. Both rows are already saved.
